# Dimensionless Creutz-ratio plot

This notebook reads the precomputed r0-scaled diagonal Creutz-ratio result and saves the plot as a PDF without a title.

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from finalized_analysis_helpers import _ensure_plotly_browser_path

CREUTZ_DIR = Path("../data/gradient_flow_wtemp_analysis/beta_2p4__L0_24__epsilon1_0__dt_0p01__b6c467be/creutz_ratio_analysis")
SUMMARY_PATH = CREUTZ_DIR / "summary.json"
PDF_PATH = CREUTZ_DIR / "dimensionless_creutz_ratios_r0_diagonal.pdf"

# Set to a list like [0.25, 0.5, 0.75, 1.0], or leave as None for all t_lat values.
SELECTED_T_LAT = None

# Set these to tuples like (xmin, xmax) / (ymin, ymax), or leave as None for automatic limits.
XLIM = None
YLIM = None

# Plot styling, matching the other Plotly PDF export notebooks.
FIGURE_WIDTH = 950
FIGURE_HEIGHT = 600
CAPSIZE = 3
MARKER_SIZE = 7
LINE_WIDTH = 1.5
PDF_MARGIN = {"l": 58, "r": 12, "t": 12, "b": 50}
COLOR_SEQUENCE = pio.templates["plotly"].layout.colorway

_ensure_plotly_browser_path()
pio.templates.default = "plotly_white"

with SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    summary = json.load(handle)

r0_over_a = summary.get("r0_over_a")
if r0_over_a is None:
    raise ValueError(f"No r0_over_a value found in {SUMMARY_PATH}")

def _is_finite(value):
    try:
        return math.isfinite(float(value))
    except (TypeError, ValueError):
        return False


def _same_t_lat(left, right):
    return math.isclose(float(left), float(right), rel_tol=1e-12, abs_tol=1e-12)


def _t_lat_is_selected(t_lat, selected_t_lat):
    if selected_t_lat is None:
        return True
    return any(_same_t_lat(t_lat, selected) for selected in selected_t_lat)


records = []
for row in summary.get("records", []):
    if not math.isclose(float(row["R_mid"]), float(row["T_mid"]), rel_tol=0.0, abs_tol=1e-12):
        continue
    if not _t_lat_is_selected(row["t_over_a2"], SELECTED_T_LAT):
        continue
    if not (_is_finite(row.get("r_hat_r0")) and _is_finite(row.get("chi_hat_r0"))):
        continue
    records.append(dict(row))

if not records:
    raise ValueError("No r0-scaled diagonal Creutz-ratio records remain after filtering.")

t_lat_values = sorted({float(row["t_over_a2"]) for row in records})
print(f"Loaded {len(records)} diagonal r0-scaled Creutz-ratio points from {SUMMARY_PATH}")
print(f"r0/a: {float(r0_over_a):.8g}")
print("t_lat values:", [f"{t_lat:g}" for t_lat in t_lat_values])


def _trace_for_t_lat(t_lat, color):
    rows = sorted(
        [row for row in records if _same_t_lat(row["t_over_a2"], t_lat)],
        key=lambda row: float(row["r_hat_r0"]),
    )
    x = np.asarray([float(row["r_hat_r0"]) for row in rows], dtype=float)
    y = np.asarray([float(row["chi_hat_r0"]) for row in rows], dtype=float)
    yerr = np.asarray(
        [float(row["chi_hat_r0_err"]) if _is_finite(row.get("chi_hat_r0_err")) else 0.0 for row in rows],
        dtype=float,
    )
    customdata = np.column_stack(
        [
            np.asarray([float(row["R_mid"]) for row in rows], dtype=float),
            np.asarray([float(row["T_mid"]) for row in rows], dtype=float),
            yerr,
            np.asarray([int(row.get("n_bootstrap_valid") or 0) for row in rows], dtype=int),
        ]
    )
    return go.Scatter(
        x=x,
        y=y,
        mode="lines+markers",
        name=f"{t_lat:g}",
        line={"color": color, "width": LINE_WIDTH},
        marker={"size": MARKER_SIZE, "color": color},
        error_y={"type": "data", "array": yerr, "visible": True, "thickness": 1, "width": CAPSIZE},
        customdata=customdata,
        hovertemplate=(
            "r/r0=%{x:.5g}<br>"
            "chi_lat (r0/a)^2=%{y:.8g}<br>"
            "Rmid=%{customdata[0]:g}<br>"
            "Tmid=%{customdata[1]:g}<br>"
            "bootstrap error=%{customdata[2]:.3g}<br>"
            "valid bootstrap samples=%{customdata[3]:.0f}"
            "<extra>t_lat=" + f"{t_lat:g}" + "</extra>"
        ),
    )


def make_dimensionless_creutz_pdf_figure(*, xlim=None, ylim=None):
    fig = go.Figure()
    for index, t_lat in enumerate(t_lat_values):
        color = COLOR_SEQUENCE[index % len(COLOR_SEQUENCE)]
        fig.add_trace(_trace_for_t_lat(t_lat, color))

    fig.update_layout(
        title=None,
        width=FIGURE_WIDTH,
        height=FIGURE_HEIGHT,
        xaxis_title="r/r0",
        yaxis_title="chi_lat (r0/a)^2",
        margin=PDF_MARGIN,
        showlegend=True,
        legend={"title": {"text": "t_lat"}, "x": 0.98, "xanchor": "right", "y": 0.98, "yanchor": "top"},
    )
    if xlim is not None:
        fig.update_xaxes(range=list(xlim))
    if ylim is not None:
        fig.update_yaxes(range=list(ylim))
    return fig


fig = make_dimensionless_creutz_pdf_figure(xlim=XLIM, ylim=YLIM)
PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.write_image(str(PDF_PATH))
print(f"Saved PDF: {PDF_PATH}")
fig


## r0-scaled fit plots

This section reads the precomputed `fits_r0/fit_t_over_a2_*.json` files, rebuilds the two-panel fit and relative-uncertainty figures, and saves selected PDFs.

In [ ]:
from plotly.subplots import make_subplots

FIT_R0_DIR = CREUTZ_DIR / "fits_r0"
FIT_R0_SUMMARY_PATH = FIT_R0_DIR / "summary.json"

# Set to None for every available fit, or to values like [0.75] / [0.0, 0.5, 1.0].
FIT_SELECTED_T_LAT = None

# The displayed figure. If None, the first selected fit is shown.
FIT_DISPLAY_T_LAT = 0.75

# Set to False to display only without rewriting PDFs.
FIT_EXPORT_PDFS = True
FIT_PDF_OUTPUT_DIR = FIT_R0_DIR
FIT_PDF_SUFFIX = ""  # e.g. "_notebook" to avoid overwriting the generated PDFs.

# Optional axis limits.
FIT_XLIM = (0,2.36)
FIT_YLIM = (1.4,1.8)
FIT_REL_YLIM = (0,0.06)

FIT_FIGURE_WIDTH = 1850
FIT_FIGURE_HEIGHT = 620
FIT_MARGIN = {"l": 85, "r": 170, "t": 80, "b": 70}
FIT_MODEL_COLORS = {
    "cspl_pol_m2": "#1f77b4",
    "cspl": "#ff7f0e",
    "pol_0_m2_m4": "#2ca02c",
    "pol_2_1_0": "#d62728",
}
FIT_RELATIVE_COLORS = {
    "total_rel": "#1f77b4",
    "sys_rel": "#2ca02c",
    "stat_rel": "#d62728",
}
FIT_RELATIVE_LABELS = {
    "total_rel": "Δ<sub>tot</sub>",
    "sys_rel": "Δ<sub>sys</sub>",
    "stat_rel": "Δ<sub>stat</sub>",
}
FIT_HIDE_SMOOTHING_RADIUS_T_LAT_VALUES = {0.75}


def _t_lat_token(t_lat):
    return f"{float(t_lat):g}".replace("-", "m").replace(".", "p")


def _load_fit_r0_summary(path=FIT_R0_SUMMARY_PATH):
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    flows = payload.get("flows", {})
    ok_entries = []
    for t_lat_text, entry in flows.items():
        if entry.get("status") != "ok" or not entry.get("json"):
            continue
        t_lat = float(t_lat_text)
        if not _t_lat_is_selected(t_lat, FIT_SELECTED_T_LAT):
            continue
        json_path = Path(entry["json"])
        if not json_path.is_absolute():
            json_path = path.parent / json_path
        ok_entries.append((t_lat, json_path, entry))
    ok_entries.sort(key=lambda item: item[0])
    if not ok_entries:
        raise ValueError(f"No selected ok fits found in {path}")
    return ok_entries


def _load_fit_payload(json_path):
    with json_path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def make_fit_r0_figure(fit, *, xlim=None, ylim=None, rel_ylim=None):
    grid = fit["grid"]
    x_grid = [float(value) for value in grid["x"]]
    eight_t_over_a2 = float(fit["eight_t_over_a2"])
    t_over_a2 = eight_t_over_a2 / 8.0
    chi_hat = "χ̂"
    x_axis_title = "r/r<sub>0</sub>"
    y_axis_title = chi_hat
    rel_y_axis_title = f"Δ{chi_hat}/{chi_hat}"
    overall_title = (
        f"Creutz-ratio interpolation at "
        f"t<sub>lat</sub> = {t_over_a2:g}"
    )
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Dimensionless Creutz ratio", "Relative interpolation uncertainty"),
        horizontal_spacing=0.10,
    )

    points = [point for point in fit["points"] if point.get("used_in_fit")] or fit["points"]
    fig.add_trace(
        go.Scatter(
            x=[float(point["x"]) for point in points],
            y=[float(point["y"]) for point in points],
            error_y={
                "type": "data",
                "array": [float(point["err"]) for point in points],
                "visible": True,
                "thickness": 1,
                "width": CAPSIZE,
            },
            mode="markers",
            name="data",
            marker={"color": "black", "size": 5},
            hovertemplate="r/r0=%{x:.5g}<br>chi_hat=%{y:.8g}<extra>data</extra>",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=x_grid,
            y=[float(value) for value in grid["average"]],
            mode="lines",
            name="average",
            line={"color": "gray", "width": 2},
            hovertemplate="r/r0=%{x:.5g}<br>Average=%{y:.8g}<extra></extra>",
        ),
        row=1,
        col=1,
    )

    for name, model in grid["models"].items():
        fig.add_trace(
            go.Scatter(
                x=x_grid,
                y=[float(value) for value in model["y"]],
                mode="lines",
                name=str(model["label"]),
                line={"color": FIT_MODEL_COLORS.get(name), "width": 1.3},
                hovertemplate="r/r0=%{x:.5g}<br>fit=%{y:.8g}<extra>" + str(model["label"]) + "</extra>",
            ),
            row=1,
            col=1,
        )

    smoothing_radius = fit.get("smoothing_radius_r_hat")
    hide_smoothing_radius = any(
        math.isclose(float(t_over_a2), value, rel_tol=1e-12, abs_tol=1e-12)
        for value in FIT_HIDE_SMOOTHING_RADIUS_T_LAT_VALUES
    )
    if smoothing_radius is not None and _is_finite(smoothing_radius) and not hide_smoothing_radius:
        y_values = [float(point["y"]) for point in points if _is_finite(point.get("y"))]
        y_values.extend(float(value) for value in grid["average"] if _is_finite(value))
        if y_values:
            fig.add_trace(
                go.Scatter(
                    x=[float(smoothing_radius), float(smoothing_radius)],
                    y=[min(y_values), max(y_values)],
                    mode="lines",
                    showlegend=False,
                    line={"color": "rgba(120,120,120,0.65)", "width": 1},
                    hoverinfo="skip",
                ),
                row=1,
                col=1,
            )

    for key in ("total_rel", "sys_rel", "stat_rel"):
        fig.add_trace(
            go.Scatter(
                x=x_grid,
                y=[float(value) for value in grid[key]],
                mode="lines",
                name=FIT_RELATIVE_LABELS[key],
                line={"color": FIT_RELATIVE_COLORS[key], "width": 1.6},
                hovertemplate="r/r0=%{x:.5g}<br>" + FIT_RELATIVE_LABELS[key] + "=%{y:.8g}<extra></extra>",
            ),
            row=1,
            col=2,
        )

    fig.update_layout(
        title={"text": overall_title, "x": 0.5, "xanchor": "center"},
        width=FIT_FIGURE_WIDTH,
        height=FIT_FIGURE_HEIGHT,
        margin=FIT_MARGIN,
        legend_title_text="Fit",
        showlegend=True,
        legend={"x": 1.01, "xanchor": "left", "y": 1.0, "yanchor": "top"},
        font={"family": "DejaVu Sans, Arial, sans-serif", "size": 15},
    )
    fig.update_xaxes(title_text=x_axis_title, title_standoff=12, row=1, col=1)
    fig.update_yaxes(title_text=y_axis_title, title_standoff=18, row=1, col=1)
    fig.update_xaxes(title_text=x_axis_title, title_standoff=12, row=1, col=2)
    fig.update_yaxes(title_text=rel_y_axis_title, title_standoff=18, rangemode="tozero", row=1, col=2)
    if xlim is not None:
        fig.update_xaxes(range=list(xlim), row=1, col=1)
        fig.update_xaxes(range=list(xlim), row=1, col=2)
    if ylim is not None:
        fig.update_yaxes(range=list(ylim), row=1, col=1)
    if rel_ylim is not None:
        fig.update_yaxes(range=list(rel_ylim), row=1, col=2)
    return fig


fit_entries = _load_fit_r0_summary()
fit_payloads = [(t_lat, _load_fit_payload(json_path), json_path) for t_lat, json_path, _entry in fit_entries]

print(f"Loaded {len(fit_payloads)} r0-scaled fit plot(s) from {FIT_R0_DIR}")
print("t_lat values:", [f"{float(t_lat):g}" for t_lat, _payload, _path in fit_payloads])

fit_figures = {}
for t_lat, payload, json_path in fit_payloads:
    fit_fig = make_fit_r0_figure(payload, xlim=FIT_XLIM, ylim=FIT_YLIM, rel_ylim=FIT_REL_YLIM)
    fit_figures[float(t_lat)] = fit_fig
    if FIT_EXPORT_PDFS:
        output_path = FIT_PDF_OUTPUT_DIR / f"fit_t_lat_{_t_lat_token(t_lat)}{FIT_PDF_SUFFIX}.pdf"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fit_fig.write_image(str(output_path))
        print(f"Saved PDF: {output_path}")

if FIT_DISPLAY_T_LAT is None:
    display_t_lat = fit_payloads[0][0]
else:
    display_t_lat = min(fit_figures, key=lambda value: abs(value - float(FIT_DISPLAY_T_LAT)))

fit_r0_fig = fit_figures[float(display_t_lat)]
print(f"Displaying t_lat = {float(display_t_lat):g}")
fit_r0_fig